# Naive Bayes from Scratch

**Interview-focused reference notebook** -- Gaussian, Multinomial, Bernoulli NB, Laplace smoothing, generative vs discriminative.

## Core Intuition

Naive Bayes is a **generative** classifier based on Bayes' theorem with a strong independence assumption:

$$P(y|\mathbf{x}) = \frac{P(\mathbf{x}|y)P(y)}{P(\mathbf{x})} \propto P(y)\prod_{j=1}^{d} P(x_j|y)$$

The "naive" assumption: features are conditionally independent given the class. This is almost never true, yet NB works surprisingly well in practice.

**Prediction:** $\hat{y} = \arg\max_k P(y=k) \prod_j P(x_j|y=k)$

In log space (for numerical stability): $\hat{y} = \arg\max_k \left[\log P(y=k) + \sum_j \log P(x_j|y=k)\right]$

**Three variants:**
- **Gaussian NB:** $P(x_j|y=k) = \mathcal{N}(\mu_{jk}, \sigma_{jk}^2)$ -- continuous features
- **Multinomial NB:** $P(x_j|y=k) \propto \theta_{jk}$ -- count features (text/bag-of-words)
- **Bernoulli NB:** $P(x_j|y=k) = \theta_{jk}^{x_j}(1-\theta_{jk})^{1-x_j}$ -- binary features

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 1. Gaussian Naive Bayes

Assumes each feature follows a Gaussian distribution within each class:

$$P(x_j | y=k) = \frac{1}{\sqrt{2\pi\sigma_{jk}^2}} \exp\left(-\frac{(x_j - \mu_{jk})^2}{2\sigma_{jk}^2}\right)$$

**Training:** Compute $\mu_{jk}$ and $\sigma_{jk}^2$ for each feature $j$ and class $k$.

In [ ]:
class GaussianNaiveBayes:
    """Gaussian Naive Bayes classifier."""
    
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        n_features = X.shape[1]
        
        # Compute class priors, means, and variances
        self.priors_ = np.zeros(n_classes)
        self.means_ = np.zeros((n_classes, n_features))
        self.vars_ = np.zeros((n_classes, n_features))
        
        for i, c in enumerate(self.classes_):
            X_c = X[y == c]
            self.priors_[i] = X_c.shape[0] / X.shape[0]
            self.means_[i] = X_c.mean(axis=0)
            self.vars_[i] = X_c.var(axis=0) + 1e-9  # small epsilon for stability
        
        return self
    
    def _log_likelihood(self, X):
        """Compute log P(x|y=k) for each class k."""
        n_classes = len(self.classes_)
        n_samples = X.shape[0]
        log_likes = np.zeros((n_samples, n_classes))
        
        for i in range(n_classes):
            # Log of Gaussian PDF (sum over features due to independence)
            log_likes[:, i] = -0.5 * np.sum(
                np.log(2 * np.pi * self.vars_[i]) + 
                (X - self.means_[i])**2 / self.vars_[i],
                axis=1
            )
        return log_likes
    
    def predict_log_proba(self, X):
        """Log posterior: log P(y|x) = log P(x|y) + log P(y) + const."""
        log_likes = self._log_likelihood(X)
        log_priors = np.log(self.priors_)
        log_posterior = log_likes + log_priors  # broadcasting
        return log_posterior
    
    def predict(self, X):
        log_posterior = self.predict_log_proba(X)
        return self.classes_[np.argmax(log_posterior, axis=1)]
    
    def predict_proba(self, X):
        log_posterior = self.predict_log_proba(X)
        # Convert to probabilities via softmax
        log_posterior -= log_posterior.max(axis=1, keepdims=True)
        probs = np.exp(log_posterior)
        return probs / probs.sum(axis=1, keepdims=True)

# Test on synthetic data
from sklearn.datasets import make_classification
X_gauss, y_gauss = make_classification(n_samples=300, n_features=2, n_redundant=0,
                                        n_informative=2, n_clusters_per_class=1, random_state=42)

gnb = GaussianNaiveBayes().fit(X_gauss, y_gauss)
acc = np.mean(gnb.predict(X_gauss) == y_gauss)
print(f"Gaussian NB accuracy: {acc:.4f}")
print(f"Class priors: {gnb.priors_}")
print(f"Class means:\n{gnb.means_.round(3)}")

## 2. Visualize Class-Conditional Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for feat_idx, ax in enumerate(axes):
    x_range = np.linspace(X_gauss[:, feat_idx].min() - 2, X_gauss[:, feat_idx].max() + 2, 200)
    
    for i, c in enumerate(gnb.classes_):
        mu = gnb.means_[i, feat_idx]
        sigma = np.sqrt(gnb.vars_[i, feat_idx])
        pdf = (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x_range - mu) / sigma)**2)
        ax.plot(x_range, pdf, linewidth=2, label=f'Class {c}: N({mu:.2f}, {sigma:.2f}^2)')
        ax.fill_between(x_range, pdf, alpha=0.2)
    
    # Add data histogram
    for c in gnb.classes_:
        ax.hist(X_gauss[y_gauss == c, feat_idx], bins=20, density=True, alpha=0.3)
    
    ax.set_xlabel(f'Feature {feat_idx}')
    ax.set_ylabel('Density')
    ax.set_title(f'Class-Conditional P(x_{feat_idx}|y)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Decision Boundary Visualization

In [ ]:
def plot_decision_boundary_nb(model, X, y, ax, title=''):
    h = 0.05
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = model.predict(np.column_stack([xx.ravel(), yy.ravel()]))
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu', s=30, alpha=0.7, edgecolors='k')
    ax.set_title(title)

fig, ax = plt.subplots(figsize=(8, 6))
plot_decision_boundary_nb(gnb, X_gauss, y_gauss, ax, 'Gaussian NB Decision Boundary')
plt.tight_layout()
plt.show()
print("GNB decision boundary is quadratic when variances differ, linear when they are equal.")

## 4. Multinomial Naive Bayes (for Text Classification)

For count/frequency features (e.g., word counts in a document):

$$P(x_j | y=k) = \theta_{jk} = \frac{N_{jk} + \alpha}{N_k + \alpha d}$$

where $N_{jk}$ = total count of feature $j$ in class $k$, $N_k$ = total count of all features in class $k$, and $\alpha$ is the Laplace smoothing parameter.

In [ ]:
class MultinomialNaiveBayes:
    """Multinomial Naive Bayes with Laplace smoothing."""
    
    def __init__(self, alpha=1.0):
        self.alpha = alpha  # Laplace smoothing
    
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        n_features = X.shape[1]
        
        self.priors_ = np.zeros(n_classes)
        self.log_probs_ = np.zeros((n_classes, n_features))
        
        for i, c in enumerate(self.classes_):
            X_c = X[y == c]
            self.priors_[i] = X_c.shape[0] / X.shape[0]
            
            # Feature counts + Laplace smoothing
            feature_counts = X_c.sum(axis=0) + self.alpha
            total_count = feature_counts.sum()
            self.log_probs_[i] = np.log(feature_counts / total_count)
        
        self.log_priors_ = np.log(self.priors_)
        return self
    
    def predict_log_proba(self, X):
        return X @ self.log_probs_.T + self.log_priors_
    
    def predict(self, X):
        return self.classes_[np.argmax(self.predict_log_proba(X), axis=1)]

## 5. Bernoulli Naive Bayes

For binary features (presence/absence):

$$P(\mathbf{x}|y=k) = \prod_j \theta_{jk}^{x_j} (1-\theta_{jk})^{1-x_j}$$

Unlike Multinomial NB, Bernoulli NB explicitly models absence of features.

In [ ]:
class BernoulliNaiveBayes:
    """Bernoulli Naive Bayes with Laplace smoothing."""
    
    def __init__(self, alpha=1.0):
        self.alpha = alpha
    
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        n_features = X.shape[1]
        
        self.priors_ = np.zeros(n_classes)
        self.theta_ = np.zeros((n_classes, n_features))  # P(x_j=1 | y=k)
        
        for i, c in enumerate(self.classes_):
            X_c = X[y == c]
            self.priors_[i] = X_c.shape[0] / X.shape[0]
            # P(x_j=1|y=k) with Laplace smoothing
            self.theta_[i] = (X_c.sum(axis=0) + self.alpha) / (X_c.shape[0] + 2 * self.alpha)
        
        self.log_priors_ = np.log(self.priors_)
        return self
    
    def predict_log_proba(self, X):
        n_classes = len(self.classes_)
        log_proba = np.zeros((X.shape[0], n_classes))
        
        for i in range(n_classes):
            log_theta = np.log(self.theta_[i])
            log_1_theta = np.log(1 - self.theta_[i])
            # P(x|y) = prod theta^x * (1-theta)^(1-x)
            log_proba[:, i] = (X @ log_theta + (1 - X) @ log_1_theta + self.log_priors_[i])
        
        return log_proba
    
    def predict(self, X):
        return self.classes_[np.argmax(self.predict_log_proba(X), axis=1)]

## 6. Text Classification Demo (Spam/Ham)

A toy bag-of-words example to show Multinomial NB in action.

In [ ]:
# Toy spam/ham dataset as bag-of-words
# Vocabulary: ['free', 'money', 'win', 'urgent', 'meeting', 'report', 'project', 'hello', 'offer', 'click']
vocab = ['free', 'money', 'win', 'urgent', 'meeting', 'report', 'project', 'hello', 'offer', 'click']

# Rows: documents, Columns: word counts
# Spam messages (label=1)
spam = np.array([
    [3, 2, 1, 2, 0, 0, 0, 0, 2, 3],  # "free free free money money win urgent urgent offer offer click click click"
    [2, 3, 2, 1, 0, 0, 0, 1, 3, 2],
    [4, 1, 3, 2, 0, 0, 0, 0, 1, 4],
    [1, 4, 1, 3, 0, 0, 0, 0, 2, 2],
    [3, 2, 2, 1, 0, 0, 0, 0, 3, 3],
    [2, 3, 1, 2, 0, 0, 0, 0, 4, 1],
    [5, 1, 2, 1, 0, 0, 0, 0, 2, 3],
    [1, 2, 3, 3, 0, 0, 0, 0, 1, 2],
])

# Ham messages (label=0)
ham = np.array([
    [0, 0, 0, 0, 3, 2, 2, 1, 0, 0],
    [0, 0, 0, 0, 2, 3, 1, 2, 0, 0],
    [0, 0, 0, 1, 1, 2, 3, 1, 0, 0],
    [0, 0, 0, 0, 2, 1, 2, 3, 0, 0],
    [0, 0, 0, 0, 3, 2, 3, 1, 0, 0],
    [0, 1, 0, 0, 1, 3, 2, 2, 0, 0],
    [0, 0, 0, 0, 2, 2, 1, 3, 0, 0],
    [1, 0, 0, 0, 3, 1, 2, 2, 0, 0],
])

X_text = np.vstack([ham, spam])
y_text = np.array([0]*8 + [1]*8)  # 0=ham, 1=spam

# Train Multinomial NB
mnb = MultinomialNaiveBayes(alpha=1.0).fit(X_text, y_text)

# Test on new "documents"
test_docs = np.array([
    [4, 3, 2, 1, 0, 0, 0, 0, 2, 3],  # looks spammy
    [0, 0, 0, 0, 2, 3, 2, 1, 0, 0],  # looks legit
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],  # ambiguous
])

predictions = mnb.predict(test_docs)
print("Multinomial NB Text Classification:")
for i, (doc, pred) in enumerate(zip(test_docs, predictions)):
    label = 'SPAM' if pred == 1 else 'HAM'
    top_words = [vocab[j] for j in np.argsort(-doc)[:3]]
    print(f"  Doc {i}: top words = {top_words} -> {label}")

# Show learned word probabilities
print("\nLearned log-probabilities per class:")
print(f"{'Word':<12} {'P(w|ham)':>10} {'P(w|spam)':>10} {'Ratio':>10}")
print("-" * 44)
for j, word in enumerate(vocab):
    lp_ham = mnb.log_probs_[0, j]
    lp_spam = mnb.log_probs_[1, j]
    print(f"{word:<12} {np.exp(lp_ham):>10.4f} {np.exp(lp_spam):>10.4f} {np.exp(lp_spam-lp_ham):>10.2f}x")

## 7. Laplace Smoothing -- Why It Matters

In [ ]:
# Demonstrate the zero-frequency problem
print("Without smoothing (alpha=0):")
print("  If a word never appears in spam training data, P(word|spam) = 0.")
print("  Then P(spam|document) = 0, no matter how many other spam words appear.")
print("  One unseen word kills the entire prediction!")
print()
print("With Laplace smoothing (alpha=1):")
print("  P(word|class) = (count + 1) / (total + vocab_size)")
print("  Unseen words get a small non-zero probability.")
print()

# Quantitative example
alphas = [0.001, 0.01, 0.1, 1.0, 10.0]
print(f"{'Alpha':<10} {'P(meeting|spam)':>18} {'P(free|spam)':>15}")
print("-" * 45)
for alpha in alphas:
    model = MultinomialNaiveBayes(alpha=alpha).fit(X_text, y_text)
    p_meeting_spam = np.exp(model.log_probs_[1, vocab.index('meeting')])
    p_free_spam = np.exp(model.log_probs_[1, vocab.index('free')])
    print(f"{alpha:<10} {p_meeting_spam:>18.6f} {p_free_spam:>15.6f}")

## 8. Gaussian NB vs Logistic Regression -- When Does NB Win?

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Scenario 1: Small data, features approximately independent
# NB should do relatively well here
rng = np.random.RandomState(42)
sizes = [20, 50, 100, 200, 500, 1000]
nb_scores = []
lr_scores = []

for n in sizes:
    # Generate data where features are independent given class (NB assumption holds)
    X_pos = rng.randn(n // 2, 5) + np.array([1, -1, 0.5, -0.5, 0])
    X_neg = rng.randn(n // 2, 5) + np.array([-1, 1, -0.5, 0.5, 0])
    X_nb = np.vstack([X_pos, X_neg])
    y_nb = np.array([1] * (n // 2) + [0] * (n // 2))
    
    gnb_model = GaussianNaiveBayes().fit(X_nb, y_nb)
    lr_model = LogisticRegression(max_iter=1000).fit(X_nb, y_nb)
    
    nb_scores.append(np.mean(gnb_model.predict(X_nb) == y_nb))
    lr_scores.append(lr_model.score(X_nb, y_nb))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sizes, nb_scores, 'bo-', linewidth=2, label='Gaussian NB')
ax.plot(sizes, lr_scores, 'rs-', linewidth=2, label='Logistic Regression')
ax.set_xlabel('Training Set Size')
ax.set_ylabel('Training Accuracy')
ax.set_title('NB vs LR: NB converges faster with less data')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("When NB wins:")
print("  - Very small training sets (NB needs fewer parameters to estimate)")
print("  - When features are approximately independent")
print("  - When you need fast training (NB is O(nd), no iteration)")
print("\nWhen LR wins:")
print("  - Larger datasets (LR is more flexible)")
print("  - Features are correlated")
print("  - Need well-calibrated probabilities")

## 9. Multi-Class Gaussian NB with Decision Boundaries

In [ ]:
from sklearn.datasets import make_blobs

# 3-class problem
X_3c, y_3c = make_blobs(n_samples=300, centers=3, n_features=2, random_state=42, cluster_std=1.5)

gnb_3 = GaussianNaiveBayes().fit(X_3c, y_3c)
acc_3 = np.mean(gnb_3.predict(X_3c) == y_3c)
print(f"3-class GNB accuracy: {acc_3:.4f}")

# Decision boundary with probability contours
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

h = 0.1
x_min, x_max = X_3c[:, 0].min() - 2, X_3c[:, 0].max() + 2
y_min, y_max = X_3c[:, 1].min() - 2, X_3c[:, 1].max() + 2
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
grid = np.column_stack([xx.ravel(), yy.ravel()])

Z = gnb_3.predict(grid).reshape(xx.shape)
axes[0].contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
scatter = axes[0].scatter(X_3c[:, 0], X_3c[:, 1], c=y_3c, cmap='viridis',
                          edgecolors='k', s=30, alpha=0.7)
axes[0].set_title('Gaussian NB Decision Regions')

# Confidence map
probs = gnb_3.predict_proba(grid)
confidence = probs.max(axis=1).reshape(xx.shape)
cs = axes[1].contourf(xx, yy, confidence, levels=20, cmap='RdYlGn', alpha=0.8)
plt.colorbar(cs, ax=axes[1], label='Confidence')
axes[1].scatter(X_3c[:, 0], X_3c[:, 1], c=y_3c, cmap='viridis',
                edgecolors='k', s=30, alpha=0.7)
axes[1].set_title('Prediction Confidence')

plt.tight_layout()
plt.show()

## 10. Sklearn Comparison

In [ ]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB

# Gaussian NB comparison
sk_gnb = GaussianNB().fit(X_gauss, y_gauss)
print("Gaussian NB:")
print(f"  Our accuracy:     {np.mean(gnb.predict(X_gauss) == y_gauss):.4f}")
print(f"  sklearn accuracy: {sk_gnb.score(X_gauss, y_gauss):.4f}")
print(f"  Our means:    {gnb.means_.round(4)}")
print(f"  sklearn means: {sk_gnb.theta_.round(4)}")

# Multinomial NB comparison
sk_mnb = MultinomialNB(alpha=1.0).fit(X_text, y_text)
print("\nMultinomial NB (text data):")
print(f"  Our accuracy:     {np.mean(mnb.predict(X_text) == y_text):.4f}")
print(f"  sklearn accuracy: {sk_mnb.score(X_text, y_text):.4f}")

# Bernoulli NB comparison
X_binary = (X_text > 0).astype(float)
bnb = BernoulliNaiveBayes(alpha=1.0).fit(X_binary, y_text)
sk_bnb = BernoulliNB(alpha=1.0).fit(X_binary, y_text)
print("\nBernoulli NB (binary text data):")
print(f"  Our accuracy:     {np.mean(bnb.predict(X_binary) == y_text):.4f}")
print(f"  sklearn accuracy: {sk_bnb.score(X_binary, y_text):.4f}")

## 11. Generative vs Discriminative Models

| Property | Generative (Naive Bayes) | Discriminative (Logistic Regression) |
|----------|-------------------------|-------------------------------------|
| Models | Joint: $P(x, y) = P(x|y)P(y)$ | Conditional: $P(y|x)$ directly |
| Assumption | Feature independence given class | Linearity of log-odds |
| Training | Estimate $P(x|y)$ and $P(y)$ | Optimize conditional likelihood |
| # Parameters | $O(d \cdot K)$ | $O(d \cdot K)$ (but learned jointly) |
| Small data | Often better (fewer effective params) | May overfit |
| Large data | Ceiling limited by independence assumption | Usually better |
| Can generate data? | Yes ($P(x|y)$ is a model of the data) | No |
| Handles missing features | Naturally (marginalize out) | Needs imputation |

## Interview Quick Reference

| Question | Answer |
|----------|--------|
| Why does NB work despite the naive assumption? | Even with wrong probability estimates, the classification (argmax) can still be correct. NB just needs to get the ranking of class posteriors right, not the exact values. Also, the independence assumption makes estimation more robust with small data. |
| Generative vs discriminative? | Generative models $P(x,y)$, can generate new data, works with small data. Discriminative models $P(y|x)$, more flexible, better with large data. NB is generative; LR is discriminative. |
| What is Laplace smoothing? | Add $\alpha$ (typically 1) to all counts to avoid zero probabilities. Without it, one unseen feature zeroes out the entire posterior. |
| When to use which NB variant? | Gaussian: continuous features. Multinomial: count features (text). Bernoulli: binary features (presence/absence). |
| NB complexity? | Training: $O(nd)$. Prediction: $O(d \cdot K)$. Very fast -- no iteration. |
| NB for text classification? | Use Multinomial NB on bag-of-words or TF-IDF features. Very strong baseline -- often competitive with more complex models. |